In [19]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime
from mlxtend.frequent_patterns import apriori, association_rules
fact = pd.read_csv("fact_sales.csv")
customers = pd.read_csv("dim_customer.csv")
products = pd.read_csv("dim_product.csv")

fact['order_date'] = pd.to_datetime(fact['order_date'])

In [20]:
import mlxtend
print(mlxtend.__version__)


0.24.0


In [21]:
snapshot_date = fact['order_date'].max() + pd.Timedelta(days=1)


In [22]:
rfm = fact.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'order_id': 'nunique',                                   # Frequency
    'total_amount': 'sum'                                    # Monetary
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']


In [23]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].astype(int)

rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [24]:
def segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return 'Champions'
    elif row['F_score'] >= 4:
        return 'Loyalists'
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'Potential Loyalist'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Hibernating'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)


In [25]:
rfm_final = rfm.merge(customers, on='customer_id', how='left')
rfm_final.head()


,customer_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment,customer_name,gender,age,city,region,signup_date
0,C001,122,1,140,1,1,1,111,Hibernating,Rahul Sharma,M,28,Bangalore,South,2023-01-10
1,C002,117,1,1619,1,1,5,115,Hibernating,Ananya Iyer,F,34,Chennai,South,2023-02-14
2,C003,99,1,1248,2,2,3,223,Hibernating,Arjun Mehta,M,41,Mumbai,West,2022-11-05
3,C004,89,1,130,2,2,1,221,Hibernating,Priya Singh,F,26,Delhi,North,2023-05-21
4,C005,81,1,1297,3,3,4,334,Others,Karthik Reddy,M,37,Hyderabad,South,2022-09-18


In [26]:
rfm_final.to_csv("rfm_customer_segments.csv", index=False)


In [27]:
basket = (
    fact.groupby(['order_id', 'product_id'])['quantity']
        .sum()
        .unstack()
        .fillna(0)
)

basket = (basket > 0).astype(int)


In [28]:
from mlxtend.frequent_patterns import apriori, association_rules

# Create basket (order-product matrix)
basket = (
    fact.groupby(['order_id', 'product_id'])['quantity']
        .sum()
        .unstack()
        .fillna(0)
)

# Convert to boolean (BEST practice)
basket = basket > 0

# Run Apriori
frequent_items = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

# Generate association rules
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.5
)

# View top rules
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']] \
    .sort_values(by='lift', ascending=False) \
    .head()


,antecedents,consequents,support,confidence,lift
0,frozenset({P001}),frozenset({P002}),0.181818,1.0,5.500000
1,frozenset({P002}),frozenset({P001}),0.181818,1.0,5.500000
8,frozenset({P008}),frozenset({P007}),0.090909,1.0,5.500000
9,frozenset({P007}),frozenset({P008}),0.090909,0.5,5.500000
3,frozenset({P009}),frozenset({P003}),0.090909,1.0,3.666667


In [29]:
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.5
)


In [30]:
product_map = products.set_index('product_id')['product_name'].to_dict()

rules['antecedents'] = rules['antecedents'].apply(
    lambda x: [product_map[i] for i in x]
)
rules['consequents'] = rules['consequents'].apply(
    lambda x: [product_map[i] for i in x]
)


In [31]:
rules_sorted = rules.sort_values(by='lift', ascending=False)
rules_sorted.to_csv("product_bundles.csv", index=False)

rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()


,antecedents,consequents,support,confidence,lift
0,[Milk 1L],[Bread],0.181818,1.0,5.500000
1,[Bread],[Milk 1L],0.181818,1.0,5.500000
8,[Butter],[Toothpaste],0.090909,1.0,5.500000
9,[Toothpaste],[Butter],0.090909,0.5,5.500000
3,[Baby Wipes],[Diapers],0.090909,1.0,3.666667


In [32]:
rules.to_csv("product_bundles.csv", index=False)


In [33]:
rfm['Segment']


0    Hibernating
1    Hibernating
2    Hibernating
3    Hibernating
4         Others
5         Others
6      Champions
7      Champions
8      Champions
9      Champions
Name: Segment, dtype: str